In part two of this project, you’ll refactor the code to follow a more structured object-oriented approach. We’ll create a Student model as a base class for managing individual students, and a StudentManagementSystem class that encapsulates all the CRUD operations, as well as the logic for handling multiple students. The data will be saved to and loaded from a JSON file using methods in the StudentManagementSystem class. Let’s break it down step-by-step:

Project Structure

Create a directory for the project:


In [1]:
!student_management_system/
!├── student.py              # Defines the Student class
!├── management_system.py     # Defines the StudentManagementSystem class
!└── main.py                  # Main script to run the system

zsh:1: no such file or directory: student_management_system/
zsh:1: command not found: ├──
zsh:1: command not found: ├──
zsh:1: command not found: └──



1.	student.py: Contains the Student model, representing individual student data and methods.
2.	management_system.py: Contains the StudentManagementSystem class for managing CRUD operations, average calculations, and data persistence.
3.	main.py: Script to interact with the student management system, running the application.

student.py - Define the Student Model

In this file, we define a Student class that handles individual student data, such as subjects and grades.



In [2]:
# student_management_system/student.py

from typing import Dict

class Student:
    def __init__(self, name: str, grades: Dict[str, int]):
        """
        Initialize a student with a name and their grades.

        Parameters:
            name (str): Name of the student.
            grades (dict): A dictionary of subjects and grades.
        """
        self.name = name
        self.grades = grades

    def add_grade(self, subject: str, grade: int) -> None:
        """Add or update a grade for a specific subject."""
        if subject not in self.grades:
            self.grades[subject] = grade
        self.grades[subject] = grade

    def get_average_grade(self) -> float:
        """Calculate the average grade for the student."""
        average_grade = sum(self.grades.values()) / len(self.grades.values())
        
        return average_grade 

    def to_dict(self) -> Dict[str, any]:
        """Convert the Student instance to a dictionary for JSON serialization."""
        return {self.name: {"grades": self.grades}}

    @staticmethod
    def from_dict(data: Dict[str, any]) -> "Student":
        """Create a Student instance from a dictionary."""
        for student, grades in data.items():
            return Student(student, grades["grades"])

In [3]:
# =================================
# Test result, with Ivy from part 1
# =================================

# Testing the constructor
Ivy = Student(name="Ivy", grades={"Math": 85, "Science": 90})
print(f"Hello, {Ivy.name}")
print(f"Your grade is {Ivy.grades}" , "\n")

# Testing get_average_grade method
print(f"The average score: {Ivy.get_average_grade()}", "\n")

# Testing to_dict method
data = Ivy.to_dict()
print("In a dictionary foramt: ", data, "\n") 

# Testing from_dict method
print("Creating Ivy from data ... ")
Ivy = Student.from_dict(data)
print(f"Hello, {Ivy.name}")
print(f"Your grade is {Ivy.grades}")


Hello, Ivy
Your grade is {'Math': 85, 'Science': 90} 

The average score: 87.5 

In a dictionary foramt:  {'Ivy': {'grades': {'Math': 85, 'Science': 90}}} 

Creating Ivy from data ... 
Hello, Ivy
Your grade is {'Math': 85, 'Science': 90}



management_system.py - Define the Student Management System

This file contains the StudentManagementSystem class, which manages a collection of Student objects and includes methods for CRUD operations, saving/loading data, and calculating averages.



In [4]:
# student_management_system/management_system.py

import json
from typing import Dict, List
from student import Student

class StudentManagementSystem:
    def __init__(self):
        """Initialize the student management system with an empty dictionary of students."""
        self.students = {}

    def add_student(self, name: str, grades: Dict[str, int]) -> bool:
        """Add a new student to the system."""
            # if the student already exists, return False
        if name in self.students:
            return False

        # if grades are not in the correct format, return False
        if not isinstance(grades, dict):
            return False

        # if subject and grade are not in the correct format, return False
        for subject, grade in grades.items():
            if not isinstance(subject, str) or not isinstance(grade, int):
                return False

        # else, add students and return True
        else:
            self.students[name] = {"grades": grades}
            return True

    def update_grade(self, name: str, subject: str, grade: int) -> bool:
        """Update a specific student's grade for a given subject."""
            # Return false if student was not found
        if name not in self.students:
            return False

        # Return false if the subject was not found
        if subject not in self.students[name]["grades"]:
            return False

        # else, update the grade and return True
        self.students[name]["grades"][subject] = grade
        return True

    def delete_student(self, name: str) -> bool:
        """Delete a student from the system."""
            # Reutrn false if the student was not found
        if name not in self.students:
            return False

        # else, delete student and return true
        del self.students[name]

        return True

    def display_records(self) -> None:
        """Display all student records with their grades and average grades."""
        # =====================================================
        # Print all student records
        # =====================================================
        print(f"All student records:\n{self.students}\n")


        # =====================================================
        # Define helper function for calculating average grades
        # =====================================================

        get_avg = lambda x: round(sum(x) / len(x), 1)

        # =====================================================
        # Print the average grade for each student
        # =====================================================

        # First get a dictionary of list of grades for each student
        grades_per_student = {student: list(student_dict["grades"].values()) for student, student_dict in self.students.items()}

        # Get average for each student from a list of grades and round it to the one decimal place
        avg_per_student = {student: get_avg(grades) for student, grades in grades_per_student.items()}

        print(f"Average grade for each student:\n{avg_per_student}\n")

        # =====================================================
        # Print the average grade for each subject
        # =====================================================
        # Get a list of grades per student first
        list_of_grades_per_student = [student_dict["grades"] for student_dict in self.students.values()]

        # Make a dictionary where keys are subjects and values are lists of grades for each student
        grades_per_subject = {}
        for grades in list_of_grades_per_student:
            for subject, grade in grades.items():
                if subject not in grades_per_subject:
                    grades_per_subject[subject] = []
                grades_per_subject[subject].append(grade)

        # Calculate average for each subject
        avg_per_subject = {subject: get_avg(grades) for subject, grades in grades_per_subject.items()}

        print(f"Average grade for each subject:\n{avg_per_subject}")

    def save_to_file(self, filename: str = "students.json") -> bool:
        """Save all students to a JSON file."""
        try:
            with open(filename, 'w', encoding='utf-8') as f:
                json.dump(self.students, f, indent=4)
            return True

        except (TypeError, OverflowError, ValueError, IOError):
            # Catches JSON serialization errors or file system I/O errors
            return False

    def load_from_file(self, filename: str = "students.json") -> bool:
        """Load student data from a JSON file."""
        try:
            with open(filename, 'r', encoding='utf-8') as file:
                self.students = json.load(file)
                return True
        except (FileNotFoundError, json.JSONDecodeError, PermissionError):
            return False

In [5]:
# ======================================
# Test result, with students from part 1
# ======================================

# ======================================
# Create students variable
# ======================================
import random
from typing import Dict, Any

# fix random seed for reproducibility
random.seed(0)

def generate_sample_data() -> Dict[str, Dict[str, Any]]:
    """
    Generate a sample dictionary of students with a random number of subjects and grades.

    Returns:
        dict: A dictionary with student names as keys, each containing their grades.
    """
    student_names = ["Alice", "Bob", "Charlie", "David", "Eva", "Frank", "Grace", "Hannah"]
    subjects = ["Math", "English", "Science", "History", "Art"]
    students = {}

    for name in student_names:
        number_of_subjects = random.randint(2, len(subjects))  # Random number of subjects
        selected_subjects = random.sample(subjects, number_of_subjects)
        grades = {subject: random.randint(50, 100) for subject in selected_subjects}
        students[name] = {"grades": grades}

    return students

# Generate and print the sample data to verify
data = generate_sample_data()
print(json.dumps(data, indent=2))

{
  "Alice": {
    "grades": {
      "History": 100,
      "Math": 69,
      "English": 80,
      "Science": 72,
      "Art": 87
    }
  },
  "Bob": {
    "grades": {
      "Art": 58,
      "English": 98,
      "History": 56
    }
  },
  "Charlie": {
    "grades": {
      "Art": 96,
      "English": 54,
      "History": 93,
      "Math": 71
    }
  },
  "David": {
    "grades": {
      "Art": 89,
      "Math": 90,
      "English": 63,
      "Science": 85,
      "History": 80
    }
  },
  "Eva": {
    "grades": {
      "Art": 96,
      "Science": 75,
      "Math": 95,
      "History": 100,
      "English": 92
    }
  },
  "Frank": {
    "grades": {
      "Art": 71,
      "History": 65
    }
  },
  "Grace": {
    "grades": {
      "Math": 65,
      "English": 59,
      "Science": 84,
      "Art": 78
    }
  },
  "Hannah": {
    "grades": {
      "Math": 82,
      "Science": 81
    }
  }
}


In [6]:
# ==========
# Test start
# ==========
# Instantiate Students instance
students = StudentManagementSystem()

# test add_student method using students data
for student, grades in data.items():
    students.add_student(name=student, grades=grades["grades"])

print("=======================================================")
print("Printing out the first three students added ")
print(json.dumps(dict(list(students.students.items())[:3]), indent=2), "\n")

# test update_grade method using students data
print("=======================================================")
print(f"Before updating grade, Alice's history grade: {students.students["Alice"]["grades"]["History"]}")
students.update_grade(name="Alice", subject="History", grade=90)
print(f"After updating grade, Alice's history grade: {students.students["Alice"]["grades"]["History"]}", "\n")
students.update_grade(name="Alice", subject="History", grade=100) # Restore original grade 

Printing out the first three students added 
{
  "Alice": {
    "grades": {
      "History": 100,
      "Math": 69,
      "English": 80,
      "Science": 72,
      "Art": 87
    }
  },
  "Bob": {
    "grades": {
      "Art": 58,
      "English": 98,
      "History": 56
    }
  },
  "Charlie": {
    "grades": {
      "Art": 96,
      "English": 54,
      "History": 93,
      "Math": 71
    }
  }
} 

Before updating grade, Alice's history grade: 100
After updating grade, Alice's history grade: 90 



True

In [7]:
# test delete_student method using students data
print("=======================================================")
import copy
copied = copy.deepcopy(students)
print("First three from original")
print(f"{json.dumps(dict(list(copied.students.items())[:3]), indent=2)}", "\n\n\n")
copied.delete_student("Bob")

First three from original
{
  "Alice": {
    "grades": {
      "History": 100,
      "Math": 69,
      "English": 80,
      "Science": 72,
      "Art": 87
    }
  },
  "Bob": {
    "grades": {
      "Art": 58,
      "English": 98,
      "History": 56
    }
  },
  "Charlie": {
    "grades": {
      "Art": 96,
      "English": 54,
      "History": 93,
      "Math": 71
    }
  }
} 





True

In [8]:
print("First three from Bob removed")
print(json.dumps(dict(list(copied.students.items())[:3]), indent=2))

First three from Bob removed
{
  "Alice": {
    "grades": {
      "History": 100,
      "Math": 69,
      "English": 80,
      "Science": 72,
      "Art": 87
    }
  },
  "Charlie": {
    "grades": {
      "Art": 96,
      "English": 54,
      "History": 93,
      "Math": 71
    }
  },
  "David": {
    "grades": {
      "Art": 89,
      "Math": 90,
      "English": 63,
      "Science": 85,
      "History": 80
    }
  }
}


In [9]:
# test display_record method using students data
students.display_records()

All student records:
{'Alice': {'grades': {'History': 100, 'Math': 69, 'English': 80, 'Science': 72, 'Art': 87}}, 'Bob': {'grades': {'Art': 58, 'English': 98, 'History': 56}}, 'Charlie': {'grades': {'Art': 96, 'English': 54, 'History': 93, 'Math': 71}}, 'David': {'grades': {'Art': 89, 'Math': 90, 'English': 63, 'Science': 85, 'History': 80}}, 'Eva': {'grades': {'Art': 96, 'Science': 75, 'Math': 95, 'History': 100, 'English': 92}}, 'Frank': {'grades': {'Art': 71, 'History': 65}}, 'Grace': {'grades': {'Math': 65, 'English': 59, 'Science': 84, 'Art': 78}}, 'Hannah': {'grades': {'Math': 82, 'Science': 81}}}

Average grade for each student:
{'Alice': 81.6, 'Bob': 70.7, 'Charlie': 78.5, 'David': 81.4, 'Eva': 91.6, 'Frank': 68.0, 'Grace': 71.5, 'Hannah': 81.5}

Average grade for each subject:
{'History': 82.3, 'Math': 78.7, 'English': 74.3, 'Science': 79.4, 'Art': 82.1}


In [10]:
# test save_to_file method
students.save_to_file()

True

In [11]:
# test load_from_file method
# Let's add Bob back to the copied and test if it works
copied.load_from_file()
print(f"Welcome back Bob! Your grade is: {copied.students["Bob"]["grades"]}")

Welcome back Bob! Your grade is: {'Art': 58, 'English': 98, 'History': 56}



main.py - Main Script to Run the System

In this script, you’ll interact with the StudentManagementSystem to manage student data.



In [12]:
# student_management_system/main.py

from management_system import StudentManagementSystem

def main():
    system = StudentManagementSystem()

    # Load existing data if available
    system.load_from_file()

    # Example operations
    system.add_student("Alice", {"Math": 85, "Science": 90})
    system.add_student("Bob", {"English": 78, "History": 82})

    system.update_grade("Alice", "Math", 92)
    system.update_grade("Bob", "History", 88)

    system.display_records()

    system.delete_student("Bob")

    # Save data
    system.save_to_file()

if __name__ == "__main__":
    main()

All student records:
{'Alice': {'grades': {'History': 100, 'Math': 92, 'English': 80, 'Science': 72, 'Art': 87}}, 'Bob': {'grades': {'Art': 58, 'English': 98, 'History': 88}}, 'Charlie': {'grades': {'Art': 96, 'English': 54, 'History': 93, 'Math': 71}}, 'David': {'grades': {'Art': 89, 'Math': 90, 'English': 63, 'Science': 85, 'History': 80}}, 'Eva': {'grades': {'Art': 96, 'Science': 75, 'Math': 95, 'History': 100, 'English': 92}}, 'Frank': {'grades': {'Art': 71, 'History': 65}}, 'Grace': {'grades': {'Math': 65, 'English': 59, 'Science': 84, 'Art': 78}}, 'Hannah': {'grades': {'Math': 82, 'Science': 81}}}

Average grade for each student:
{'Alice': 86.2, 'Bob': 81.3, 'Charlie': 78.5, 'David': 81.4, 'Eva': 91.6, 'Frank': 68.0, 'Grace': 71.5, 'Hannah': 81.5}

Average grade for each subject:
{'History': 87.7, 'Math': 82.5, 'English': 74.3, 'Science': 79.4, 'Art': 82.1}



### Summary

In this refactored project, we introduced object-oriented principles:

1.	Student Class: Represents an individual student with methods to manage grades and calculate averages.
2.	StudentManagementSystem Class: Encapsulates CRUD operations, average calculations, and data persistence for a collection of students.
3.	main.py: The main script to run the program and interact with the StudentManagementSystem.

This structure makes the code modular, reusable, and easier to maintain.

### How to Submit the Project

1. Create a Jupyter Notebook file on your Google Drive.
2. Write your solution in the notebook.
3. Share access with your mentor (or provide a link with editing rights).
4. Submit the link to the notebook file or the folder as your solution for this lesson.